# Cleaning up a dataset #
---
**When there's a new dataset to work with, the data frequently has mistakes. You have to clean it up before it can be properly used.**  
Let's practice doing this with a dataset containing a list of Gen I pokemon and their attributes.

In [1]:
import pandas as pd
from functools import reduce

pokemon = pd.read_csv('pokemon.csv')
pokemon

,Name,Height (in),Weight (lbs),Type,Weaknesses
0,Bulbasaur,28,15.20,"Grass, Poison","Fire, Psychic, Flying, Ie"
1,Ivysaur,39,28.70,"Grass, Poison","Fire, Psychic, Flying, Ice"
2,Venusaur,79,220.50,"Grass, Poison","Fire, Psychic, Flying, Ice"
3,Charmander,24,18.70,Fire,"Water, Ground, Rock"
4,Charmeleon,43,41.90,Fire,"Water, Ground, Rock"
...,...,...,...,...,...
149,Dratini,71,7.30,Dragon,"Fairy, Ice, Dragon"
150,Dragonair,157,36.40,Dragon,"Fairy, Ice, Dragon"
151,Dragonite,87,463.00,"Dragon, Flying","Fairy, Dragon, Ice, Rock"
152,Mewtwo,79,269.00,Psychic,"Ghost, Dark, Bug"


In [2]:
pokemon.shape

(154, 5)

In [3]:
pokemon.info()

<class 'pandas.DataFrame'>
RangeIndex: 154 entries, 0 to 153
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Name          154 non-null    str  
 1   Height (in)   153 non-null    str  
 2   Weight (lbs)  153 non-null    str  
 3   Type          152 non-null    str  
 4   Weaknesses    154 non-null    str  
dtypes: str(5)
memory usage: 6.1 KB


**All the data types are strings. That's not good. Some of these should be numbers. There must be at least one string in each column.**

In [4]:
pokemon.describe()

,Name,Height (in),Weight (lbs),Type,Weaknesses
count,154,153,153,152,154
unique,151,32,110,50,49
top,Blastoise,39,66.10,Water,"Grass, Electric"
freq,2,19,6,15,16


---
## Some preliminaries: a search function and a shift function and elementwise and/or functions for lists ##
**I'd like to be able to search for multiple things across the entire dataframe at once.**  
**I'd like to be able to fix the order of the pokemon.**  
**I'd like to be able to do elementwise and/or on lists.**

In [5]:
def search(df, ll, exclude=None):
    """
    df is a dataframe, ll is a list of search terms
    this function will search every column of the dataframe for any occurrences of any of the search
    terms and return a series that can be used as index for slicing a dataframe
    You can search for both numbers and strings in a single list of search terms
    the function will search the dataframe for anywhere your search strings are contained (containment)
    the function will search the dataframe for anywhere your search numbers occur (equality)
    You can also include a list of terms to exclude. it should be the same length as ll
    It should also be paired up with ll elementwise
    for example, ll=['Ic', 'Gras'] and exclude=['Ice', 'Grass'] will find instances where
    'Ic' is present, but 'Ice' isn't; and where 'Gras' is present, but 'Grass' isn't
    this way you can find the typos 'Ic' and 'Gras' without finding all the correct ones with 'Ice'
    and 'Grass'
    """
    # this returns a nested list where the outside dim=number of search terms in ll,
    # middle dim= number of columns in df
    # inside dim=number of rows in df
    if exclude == None:
        ind = [[[test in str(string) if type(test)==str else test==string for string in df[col].values]
                for col in df.columns]
               for test in ll]
    else:
        ind = [[[(ll[i] in str(string) and exclude[i] not in str(string)) if type(ll[i])==str else ll[i]==string for string in df[col].values]
                for col in df.columns]
               for i in range(len(ll))]
    # now we perform element wise or across the columns and search terms to give of the rows where
    # any of the search terms exist
    ind = [reduce(elementwise_or, ind[i]) for i in range(len(ind))] # for each search term, collate the columns
    ind = reduce(elementwise_or, ind) # collate the search terms
    return ind


def move_and_shift(df, start, target):
    """
    This function takes the index of the row you'd like to move and the index to which you'd like
    to move it
    the function will shift all of the rows in between up or down one to make space
    46 start
    22 target
    """
    insert = df.iloc[start]
    
    if start > target: # if you want to move the row up and shift the rest down:
        df.iloc[target+1:start+1] = df.iloc[target:start]
    if start < target: # if you want to move the row down and shift the rest up:
        df.iloc[start:target] = df.iloc[start+1:target+1]
    
    df.iloc[target] = insert


def elementwise_or(list1, list2):
    return [list1[i] or list2[i] for i in range(len(list1))]


def elementwise_and(list1, list2):
    return [list1[i] and list2[i] for i in range(len(list1))]


---
## Data cleaning ##
**Now that we've seen an overview of the data, it's time to clean it up:**
- Duplicates
- Missing data
- Incorrect data types (e.g., strings where they don't belong)
- Bad data formatting (e.g., using the wrong case or punctuation)
- Typos
- Hanging Whitespace

---
### Duplicates ###
**Let's find the duplicates. Set `keep=False` to see them all, otherwise you'll only see one of each, not two.**

In [6]:
# importantly, if we leave the positional argument empty, it looks for perfect duplicates,
# that is, every column is the same
# we just want to catch duplicate names
pokemon[pokemon.duplicated('Name', keep=False)]

,Name,Height (in),Weight (lbs),Type,Weaknesses
8,Blastoise,63 inches,188.50,Water,"Grass, Electric"
9,Blastoise,63,188.50,Water,"Grass, Electric"
23,Ekans,79,15.20,Poison,"Psychic, Ground"
48,Ekans,79,15.20,Poison,"Psychic, Ground"
84,Farfetch'd,31,33.10,"Normal, Flying","Electric, Ice, Rock"
91,Farfetch'd,31,33.10,"Normal, Flying","Electric, Ice, Rock"


**Now let's drop them. It looks like they're identical except the first Blastoise, which has a data type issue. So let's keep the last ones of each and save it inplace and reset our indices so that they don't skip.**

In [7]:
pokemon.drop_duplicates('Name', keep='last', inplace=True, ignore_index=True)
pokemon

,Name,Height (in),Weight (lbs),Type,Weaknesses
0,Bulbasaur,28,15.20,"Grass, Poison","Fire, Psychic, Flying, Ie"
1,Ivysaur,39,28.70,"Grass, Poison","Fire, Psychic, Flying, Ice"
2,Venusaur,79,220.50,"Grass, Poison","Fire, Psychic, Flying, Ice"
3,Charmander,24,18.70,Fire,"Water, Ground, Rock"
4,Charmeleon,43,41.90,Fire,"Water, Ground, Rock"
...,...,...,...,...,...
146,Dratini,71,7.30,Dragon,"Fairy, Ice, Dragon"
147,Dragonair,157,36.40,Dragon,"Fairy, Ice, Dragon"
148,Dragonite,87,463.00,"Dragon, Flying","Fairy, Dragon, Ice, Rock"
149,Mewtwo,79,269.00,Psychic,"Ghost, Dark, Bug"


In [8]:
pokemon[pokemon.duplicated('Name', keep=False)]

,Name,Height (in),Weight (lbs),Type,Weaknesses


**Tah-dah. No more duplicates. Now let's make sure they're in the right order by pokedex number.**
- Blastoise is #9 -> index 8
- Ekans is #23 -> index 22
- Farfetch'd is #83 -> index 82

In [9]:
pokemon[search(pokemon, ["Blastoise", "Ekans", "Farfetch'd"])]

,Name,Height (in),Weight (lbs),Type,Weaknesses
8,Blastoise,63,188.50,Water,"Grass, Electric"
46,Ekans,79,15.20,Poison,"Psychic, Ground"
88,Farfetch'd,31,33.10,"Normal, Flying","Electric, Ice, Rock"


In [10]:
move_and_shift(pokemon, 46, 22)
move_and_shift(pokemon, 88, 82)
pokemon[search(pokemon, ["Blastoise", "Ekans", "Farfetch'd"])]

,Name,Height (in),Weight (lbs),Type,Weaknesses
8,Blastoise,63,188.50,Water,"Grass, Electric"
22,Ekans,79,15.20,Poison,"Psychic, Ground"
82,Farfetch'd,31,33.10,"Normal, Flying","Electric, Ice, Rock"


---
### Missing data ###
**Missing data could appear in any column, and it can appear as a literal blank, or else `None` or `NaN`, etc.**  
**`isna` will find it.**

In [11]:
pokemon[(pd.isna(pokemon['Height (in)']) |
         pd.isna(pokemon['Weight (lbs)']) |
         pd.isna(pokemon['Type']) |
         pd.isna(pokemon['Weaknesses'])
        )]

,Name,Height (in),Weight (lbs),Type,Weaknesses
41,Golbat,NaN,NaN,"Poison, Flying","Psychic, Electric, Ice, Rock"
66,Machoke,59,155.40,NaN,"Psychic, Flying, Fairy"
85,Seel,43,198.40,NaN,"Grass, Electric"


**I checked bulbapedia. Let's enter the correct information:**  
Note that I have to enter the numbers as strings at this phase of cleaning because I haven't solved the issue of all the data types being strings yet.

In [12]:
pokemon.at[41,'Height (in)'] = '63'
pokemon.at[41,'Weight (lbs)'] = '121.3'
pokemon.at[66,'Type'] = 'Fighting'
pokemon.at[85,'Type'] = 'Water'

pokemon.iloc[[41, 66, 85]]

,Name,Height (in),Weight (lbs),Type,Weaknesses
41,Golbat,63,121.3,"Poison, Flying","Psychic, Electric, Ice, Rock"
66,Machoke,59,155.40,Fighting,"Psychic, Flying, Fairy"
85,Seel,43,198.40,Water,"Grass, Electric"


**Tah-dah.**

**Another way data could be missing is if they entered 'Unknown' or 'unknown.'**

In [13]:
pokemon[search(pokemon,['Unknown', 'unknown'])]

,Name,Height (in),Weight (lbs),Type,Weaknesses
34,Unknown,24,16.50,Fairy,"Steel, Posion"
117,Goldeen,24,33.10,Unknown,"Grass, Electric"
150,Mew,Unknown,8.80,Psychic,"Ghost, Dark, Bug"


**Now let's fill these in. Back to Bulbapedia.**
- Still entering them as strings.

In [14]:
pokemon.at[34, 'Name'] = 'Clefairy'
pokemon.at[117, 'Type'] = 'Water'
pokemon.at[150, 'Height (in)'] = '16'

pokemon.iloc[[34, 117, 150]]

,Name,Height (in),Weight (lbs),Type,Weaknesses
34,Clefairy,24,16.50,Fairy,"Steel, Posion"
117,Goldeen,24,33.10,Water,"Grass, Electric"
150,Mew,16,8.80,Psychic,"Ghost, Dark, Bug"


---
### Incorrect data types ###
Now let's make height and weight into numbers like they're supposed to be.

In [15]:
pokemon['Height (in)'] = pd.to_numeric(pokemon['Height (in)'], errors='coerce').astype('Int64')
pokemon['Weight (lbs)'] = pd.to_numeric(pokemon['Weight (lbs)'], errors='coerce').astype('Float64')

**Now we've turned the numbers into numbers and the other stuff into NaN, so let's go find those NaNs and fix them like we did before.**

In [16]:
pokemon[(pd.isna(pokemon['Height (in)']) |
         pd.isna(pokemon['Weight (lbs)'])
        )]

,Name,Height (in),Weight (lbs),Type,Weaknesses
23,Arbok,138,<NA>,Posion,"Psychic, Ground"
28,Nidoran (f),<NA>,15.4,Poison,"Psychic, Ground"
35,Clefable,<NA>,88.2,Fairy,"Steel, Posion"
74,Graveler,<NA>,231.5,"Rock, Ground","Steel, Fighting, Water, Ice, Grass, Ground"
76,Ponyta,39,<NA>,Fie,"Water, Ground, Rock"
126,Pinsir,<NA>,121.3,Bug,"Fire, Flying, Rock"
142,Snorlax,83,<NA>,Normal,Fighting
145,Moltres,79,<NA>,"Fire, Flying","Water, Electric, Rock"


**Back to Bulbapedia.**

In [17]:
pokemon.at[28, 'Height (in)'] = 12 * 1 + 4
pokemon.at[35, 'Height (in)'] = 12 * 4 + 3
pokemon.at[74, 'Height (in)'] = 12 * 3 + 3
pokemon.at[126, 'Height (in)'] = 12 * 4 + 11
pokemon.at[23, 'Weight (lbs)'] = 143.3
pokemon.at[76, 'Weight (lbs)'] = 66.1
pokemon.at[142, 'Weight (lbs)'] = 1014.1
pokemon.at[145, 'Weight (lbs)'] = 132.3

pokemon.iloc[[23, 28, 35, 74, 76, 126, 142, 145]]

,Name,Height (in),Weight (lbs),Type,Weaknesses
23,Arbok,138,143.3,Posion,"Psychic, Ground"
28,Nidoran (f),16,15.4,Poison,"Psychic, Ground"
35,Clefable,51,88.2,Fairy,"Steel, Posion"
74,Graveler,39,231.5,"Rock, Ground","Steel, Fighting, Water, Ice, Grass, Ground"
76,Ponyta,39,66.1,Fie,"Water, Ground, Rock"
126,Pinsir,59,121.3,Bug,"Fire, Flying, Rock"
142,Snorlax,83,1014.1,Normal,Fighting
145,Moltres,79,132.3,"Fire, Flying","Water, Electric, Rock"


In [18]:
pokemon.dtypes

Name                str
Height (in)       Int64
Weight (lbs)    Float64
Type                str
Weaknesses          str
dtype: object

**The correct values have been entered and the datatype is now correct.**

---
### Bad data formatting and/or typos ###
We expect most of these to occur in the Type and Weaknesses columns. Thankfully those columns should have a relatively small number of different values. So let's get a short list of unique values and just read them for errors.

In [19]:
# basically what i'm going to do is get the list of unique types,
# and then join and split them from the whitespace and commas until i can see all the types
types = pokemon.Type.unique()
types = ','.join(types)
types = types.split()
types = ','.join(types)
types = types.split(',')
types = set(types)
types.remove('')
types

{'Bug',
 'Dragon',
 'Electric',
 'Fairy',
 'Fie',
 'Fighting',
 'Fire',
 'Flying',
 'Ghost',
 'Grass',
 'Ground',
 'Ice',
 'Normal',
 'Poison',
 'Posion',
 'Psychic',
 'Rock',
 'Steel',
 'Water'}

**Fie, and Posion are not correct. Let's find them and destroy them.**

In [20]:
pokemon[search(pokemon, ['Fie', 'Posion'])]

,Name,Height (in),Weight (lbs),Type,Weaknesses
23,Arbok,138,143.3,Posion,"Psychic, Ground"
29,Nidorina,31,44.1,Posion,"Psychic, Ground"
30,Nidoqueen,51,132.3,"Posion, Ground","Water, Psychic, Ice, Ground"
31,Nidoran (m),20,19.8,Posion,"Psychic, Ground"
34,Clefairy,24,16.5,Fairy,"Steel, Posion"
35,Clefable,51,88.2,Fairy,"Steel, Posion"
38,Jigglypuff,20,12.1,"Normal, Fairy","Steel, Posion"
39,Wigglytuff,39,26.5,"Normal, Fairy","Steel, Posion"
76,Ponyta,39,66.1,Fie,"Water, Ground, Rock"


In [21]:
pokemon.at[23, 'Type'] = 'Poison'
pokemon.at[29, 'Type'] = 'Poison'
pokemon.at[30, 'Type'] = 'Poison, Ground'
pokemon.at[31, 'Type'] = 'Poison'
pokemon.at[34, 'Weaknesses'] = 'Steel, Poison'
pokemon.at[35, 'Weaknesses'] = 'Steel, Poison'
pokemon.at[38, 'Weaknesses'] = 'Steel, Poison'
pokemon.at[39, 'Weaknesses'] = 'Steel, Poison'
pokemon.at[76, 'Type'] = 'Fire'

pokemon.iloc[[23, 29, 30, 31, 34, 35, 38, 39, 76]]

,Name,Height (in),Weight (lbs),Type,Weaknesses
23,Arbok,138,143.3,Poison,"Psychic, Ground"
29,Nidorina,31,44.1,Poison,"Psychic, Ground"
30,Nidoqueen,51,132.3,"Poison, Ground","Water, Psychic, Ice, Ground"
31,Nidoran (m),20,19.8,Poison,"Psychic, Ground"
34,Clefairy,24,16.5,Fairy,"Steel, Poison"
35,Clefable,51,88.2,Fairy,"Steel, Poison"
38,Jigglypuff,20,12.1,"Normal, Fairy","Steel, Poison"
39,Wigglytuff,39,26.5,"Normal, Fairy","Steel, Poison"
76,Ponyta,39,66.1,Fire,"Water, Ground, Rock"


**Tah-dah.**  
**Now let's do the same thing for weaknesses.**

In [22]:
# basically what i'm going to do is get the list of unique types,
# and then join and split them from the whitespace and commas until i can see all the types
weak = ','.join(pokemon.Weaknesses.unique())
weak = weak.split()
weak = ','.join(weak)
weak = weak.split(',')
weak = set(weak)
weak.remove('')
weak

{'-',
 'Bug',
 'Dark',
 'Dragon',
 'Electric',
 'Fairy',
 'Fighting',
 'Fire',
 'Flying',
 'Flyng',
 'Ghost',
 'Gras',
 'Grass',
 'Ground',
 'Ic',
 'Ice',
 'Ie',
 'Poison',
 'Psychic',
 'Rock',
 'Steel',
 'Water'}

**Flyng, Gras, Ic, Ie are incorrect. Also, there shouldn't be any dashes. Commas only. Let's go get them.**

In [23]:
pokemon[search(pokemon, ['Flyng', 'Gras,', 'Ic', 'Ie', '-'])]

,Name,Height (in),Weight (lbs),Type,Weaknesses
0,Bulbasaur,28,15.2,"Grass, Poison","Fire, Psychic, Flying, Ie"
1,Ivysaur,39,28.7,"Grass, Poison","Fire, Psychic, Flying, Ice"
2,Venusaur,79,220.5,"Grass, Poison","Fire, Psychic, Flying, Ice"
6,Squirtle,20,19.8,Water,"Gras, Electric"
11,Butterfree,43,70.5,"Bug, Flying","Fire, Flying, Electric, Ice, Rock"
15,Pidgey,12,4.0,"Normal, Flying","Electric, Ice, Rock"
16,Pidgeotto,43,66.1,"Normal, Flying","Electric, Ic, Rock"
17,Pidgeot,59,87.1,"Normal, Flying","Electric, Ice, Rock"
20,Spearow,12,4.4,"Normal, Flying","Electric, Ie, Rock"
21,Fearow,47,83.8,"Normal, Flying","Electric, Ice, Rock"


**Well that's not good. As we might have suspected, searching for 'Ic' returns every ice type because 'Ic' is contained in 'Ice'. Similar for 'Gras'. Let's just search for the other terms and hold off on 'Ic' and 'Gras'.**

In [24]:
pokemon[search(pokemon, ['Flyng', 'Ie', '-'])]

,Name,Height (in),Weight (lbs),Type,Weaknesses
0,Bulbasaur,28,15.2,"Grass, Poison","Fire, Psychic, Flying, Ie"
20,Spearow,12,4.4,"Normal, Flying","Electric, Ie, Rock"
53,Psyduck,31,43.2,Water,Grass - Electric
56,Primeape,39,70.5,Fighting,"Psychic, Flyng, Fairy"
64,Alakazam,59,-105.8,Psychic,"Ghost, Dark, Bug"
101,Exeggcute,16,5.5,"Grass, Psychic",Ghost - Fire - Flying - Ice - Dark - Poison - Bug
107,Lickitung,47,-144.4,Normal,Fighting


**There we go. And we also caught some bad negative numbers too! Let's fix those while we're at it.**

In [25]:
pokemon.at[0, 'Weaknesses'] = 'Fire, Psychic, Flying, Ice'
pokemon.at[20, 'Weaknesses'] = 'Electric, Ice, Rock'
pokemon.at[53, 'Weaknesses'] = 'Grass, Electric'
pokemon.at[56, 'Weaknesses'] = 'Psychic, Flying, Fairy'
pokemon.at[64, 'Weight (lbs)'] = 105.8
pokemon.at[101, 'Weaknesses'] = 'Ghost, Fire, Flying, Ice, Dark, Poison, Bug'
pokemon.at[107, 'Weight (lbs)'] = 144.4

pokemon.iloc[[0, 20, 53, 56, 64, 101, 107]]

,Name,Height (in),Weight (lbs),Type,Weaknesses
0,Bulbasaur,28,15.2,"Grass, Poison","Fire, Psychic, Flying, Ice"
20,Spearow,12,4.4,"Normal, Flying","Electric, Ice, Rock"
53,Psyduck,31,43.2,Water,"Grass, Electric"
56,Primeape,39,70.5,Fighting,"Psychic, Flying, Fairy"
64,Alakazam,59,105.8,Psychic,"Ghost, Dark, Bug"
101,Exeggcute,16,5.5,"Grass, Psychic","Ghost, Fire, Flying, Ice, Dark, Poison, Bug"
107,Lickitung,47,144.4,Normal,Fighting


**Now let's go back and find and fix 'Ic' and 'Gras', excluding the good ones.**

In [26]:
pokemon[search(pokemon, ['Ic', 'Gras'], exclude=['Ice', 'Grass'])]

,Name,Height (in),Weight (lbs),Type,Weaknesses
6,Squirtle,20,19.8,Water,"Gras, Electric"
16,Pidgeotto,43,66.1,"Normal, Flying","Electric, Ic, Rock"


In [27]:
pokemon.at[6, 'Weaknesses'] = 'Grass, Electric'
pokemon.at[16, 'Weaknesses'] = 'Electric, Ice, Rock'

pokemon.iloc[[6, 16]]

,Name,Height (in),Weight (lbs),Type,Weaknesses
6,Squirtle,20,19.8,Water,"Grass, Electric"
16,Pidgeotto,43,66.1,"Normal, Flying","Electric, Ice, Rock"


---
### Hanging Whitespace ###
One last thing.

In [28]:
pokemon = pokemon.map(lambda x: x.strip() if isinstance(x, str) else x)

# if we had striped the whitespace earlier, before we turned the string columns into number columns,
# we could've done this: (commented out so it doesn't run)

#for col in pokemon.columns:
#    pokemon[col] = pokemon[col].str.strip()

---
## The end. ##
And that should do it. Let's save our changes to a .csv and go look at it in excel to verify.

In [29]:
pokemon.to_csv('clean_pokemon.csv', index=False)